# SobraBoa — IA de Triagem de Alimentos

**Projeto:** Global Solution 2026/2 — CitizenTech  
**Disciplina:** Disruptive Architectures: IoT, IoB & Generative IA  
**Grupo:** Felipe Kirschner Modesto (RM561810), João Victor L. O. Resende (RM565139), Pedro Vaz Ferreira (RM566551), Vitor Dias dos Santos (RM565422)

## O que essa IA faz

O SobraBoa é uma plataforma de redistribuição de excedentes alimentares. Quando um doador (padaria, feira, restaurante) vai publicar uma doação, ele tira foto do alimento. Essa IA:

1. **Analisa a imagem** (Visão Computacional) e classifica o alimento
2. **Retorna um JSON estruturado** com categoria, validade estimada, necessidade de refrigeração e se está próprio para doação
3. **Abre um chat contextual** (IA Generativa) onde o doador pode tirar dúvidas sobre aquele alimento específico

O contexto fixo de categorias e regras do sistema funciona como um **RAG básico** — a IA sempre responde dentro das regras da plataforma.

## Arquitetura

```
Foto do alimento
      │
      ▼
Gemini 3.5 Flash-Lite  ◄─── System Prompt (categorias + regras SobraBoa = RAG básico)
      │
      ├──► JSON estruturado (categoria, validade, refrigeração, aprovação)
      │
      └──► Chat conversacional com contexto do alimento analisado
```

In [9]:
# Célula 1 — Instalação das dependências
!pip install -q google-generativeai requests Pillow

In [10]:
# Célula 2 — Imports e configuração do modelo
import google.generativeai as genai
import requests
from PIL import Image
from io import BytesIO
import json
from google.colab import userdata

# Adiciona a chave no Secrets do Colab com o nome GEMINI_API_KEY
# (ícone de chave no menu lateral esquerdo)
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)

modelo = genai.GenerativeModel(
    model_name="gemini-3.5-flash-lite",
    system_instruction="""
Você é o assistente de triagem de alimentos do SobraBoa, uma plataforma brasileira
de redistribuição de excedentes alimentares que conecta pequenos comércios
(padarias, feiras, restaurantes) a ONGs e cozinhas comunitárias.

Seu papel é analisar alimentos que doadores desejam publicar na plataforma e retornar
uma avaliação estruturada com base nas seguintes categorias cadastradas no sistema:
- Pães e Confeitaria
- Hortifrúti (frutas, legumes, verduras)
- Cozinhados e Refeições Prontas
- Laticínios
- Carnes e Proteínas
- Bebidas
- Mercearia (enlatados, grãos, massas)
- Outros

Regras de negócio que você deve aplicar:
1. Alimentos visivelmente estragados (mofo, deterioração, coloração anormal) NÃO são
   próprios para doação e devem ser recusados com explicação clara.
2. Alimentos que requerem refrigeração (laticínios, carnes, cozinhados com proteína)
   devem ser marcados com alerta de cadeia fria.
3. A validade estimada deve ser conservadora — prefira subestimar a superestimar.
4. Oriente o doador sobre embalagem e transporte quando necessário.
5. Base legal: Lei nº 14.016/2020 (Brasil) permite doação de alimentos excedentes
   por estabelecimentos comerciais desde que próprios para consumo.

Sempre responda em português brasileiro, com linguagem simples e direta.
"""
)

print("✅ Modelo configurado com sucesso!")

✅ Modelo configurado com sucesso!


In [15]:
import urllib.request
from PIL import Image
from io import BytesIO
import json

def carregar_imagem_url(url):
    req = urllib.request.Request(url, headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
    })
    with urllib.request.urlopen(req, timeout=15) as r:
        dados = r.read()
    imagem = Image.open(BytesIO(dados)).convert("RGB")
    return imagem

def analisar_alimento(imagem, descricao_extra=""):
    prompt = "Analise este alimento para doação na plataforma SobraBoa e retorne apenas o JSON de avaliação."
    if descricao_extra:
        prompt += f"\n\nInformação adicional do doador: {descricao_extra}"
    resposta = modelo.generate_content([prompt, imagem])
    texto = resposta.text.strip()
    if "```" in texto:
        texto = texto.split("```")[1]
        if texto.startswith("json"):
            texto = texto[4:]
    return json.loads(texto.strip())

def exibir_resultado(resultado, titulo=""):
    if titulo:
        print(f"\n{'='*50}")
        print(f"  {titulo}")
        print("="*50)
    for chave, valor in resultado.items():
        print(f"  {chave}: {valor}")

## Testes de Visão Computacional

Três cenários para validar a IA:
- **Teste 1:** Alimento bom → deve ser aprovado
- **Teste 2:** Alimento estragado → deve ser recusado
- **Teste 3:** Alimento com cadeia fria → deve ser aprovado com alerta de refrigeração

In [16]:
# Célula 4 — Teste 1: Alimento bom (pão fresco)
url_pao = "https://upload.wikimedia.org/wikipedia/commons/3/33/Fresh_made_bread_05.jpg"

print("Carregando imagem...")
imagem_pao = carregar_imagem_url(url_pao)
print("Analisando com Gemini...")

resultado_pao = analisar_alimento(imagem_pao, descricao_extra="Saiu do forno há 2 horas")
exibir_resultado(resultado_pao, "Pão francês fresco")

Carregando imagem...
Analisando com Gemini...

  Pão francês fresco
  aprovado: True
  categoria: Pães e Confeitaria
  nome_alimento: Pão integral fatiado
  validade_estimada: 2 a 3 dias
  alerta_cadeia_fria: False
  orientacao_embalagem: Acondicionar em saco de papel ou plástico limpo e vedado para manter a maciez e evitar contaminação.
  motivo: O alimento apresenta excelente aspecto visual, sem sinais de mofo ou deterioração. Perfeito para doação conforme a Lei nº 14.016/2020.


In [22]:
# Célula 5 — Teste 2: Alimento impróprio (mofado)
url_mofado = "https://upload.wikimedia.org/wikipedia/commons/6/6e/Moldy_nectarines.jpg"

print("Carregando imagem...")
imagem_mofada = carregar_imagem_url(url_mofado)
print("Analisando com Gemini...")

resultado_mofado = analisar_alimento(imagem_mofada)
exibir_resultado(resultado_mofado, "Alimento com mofo visível")

Carregando imagem...
Analisando com Gemini...

  Alimento com mofo visível
  aprovado: False
  categoria: Hortifrúti (frutas, legumes, verduras)
  motivo_recusa: Alimentos visivelmente estragados com presença evidente de mofos branco e preto, indicando avançado estado de deterioração e contaminação. Impróprios para consumo humano conforme regras da plataforma e da Lei nº 14.016/2020.
  alerta_cadeia_fria: False
  validade_estimada_dias: 0
  orientacoes: Descarte os frutos mofados adequadamente. Caso haja frutos totalmente íntegros e sem contato direto com o mofo na mesma bacia, verifique-os individualmente antes de qualquer aproveitamento interno, mas estes exemplares específicos não podem ser doados.


In [25]:
# Célula 6 — Teste 3: Alimento com cadeia fria (laticínio)
url_queijo = "https://upload.wikimedia.org/wikipedia/commons/0/0e/Milk_glass.jpg"

print("Carregando imagem...")
imagem_queijo = carregar_imagem_url(url_queijo)
print("Analisando com Gemini...")

resultado_queijo = analisar_alimento(imagem_queijo, descricao_extra="Queijos artesanais, foram refrigerados até agora")
exibir_resultado(resultado_queijo, "Laticínio — alerta de cadeia fria")

Carregando imagem...
Analisando com Gemini...

  Laticínio — alerta de cadeia fria
  aprovado: True
  categoria: Laticínios
  nome_sugerido: Leite Fresco
  alerta_cadeia_fria: True
  validade_estimada: 2 dias
  instrucoes_transporte: O transporte deve ser feito obrigatoriamente em recipiente refrigerado ou bolsa térmica para manter a temperatura adequada.
  base_legal: Doação autorizada conforme a Lei nº 14.016/2020, respeitando os cuidados necessários com a conservação e segurança alimentar.


## Chat Conversacional (IA Generativa)

Após a análise da imagem, o doador pode tirar dúvidas sobre aquele alimento específico. O chat usa o JSON da análise como contexto — é aqui que entra a IA generativa conversacional integrada ao fluxo da solução.

In [26]:
# Célula 7 — Chat contextual com base no alimento analisado
# Usa o resultado do Teste 1 (pão fresco) como contexto

contexto_alimento = json.dumps(resultado_pao, ensure_ascii=False, indent=2)
historico_chat = []


def chat_assistente(mensagem_usuario):
    """Chat conversacional com contexto do alimento já analisado."""
    historico_texto = ""
    for msg in historico_chat:
        prefixo = "DOADOR" if msg["role"] == "user" else "ASSISTENTE"
        historico_texto += f"\n{prefixo}: {msg['texto']}"

    prompt = f"""
O doador já analisou o seguinte alimento na plataforma SobraBoa:
{contexto_alimento}

Histórico da conversa:{historico_texto if historico_texto else ' (início da conversa)'}

DOADOR: {mensagem_usuario}

Responda de forma direta e prática. Máximo 3 linhas.
"""

    resposta = modelo.generate_content(prompt)
    texto_resposta = resposta.text.strip()

    historico_chat.append({"role": "user", "texto": mensagem_usuario})
    historico_chat.append({"role": "model", "texto": texto_resposta})

    return texto_resposta


print("Chat iniciado com contexto do pão fresco (Teste 1)")
print("=" * 55)

perguntas = [
    "Posso embalar em saco plástico comum?",
    "O receptor pode buscar em até 3 horas?",
    "Preciso colocar alguma etiqueta de identificação?"
]

for pergunta in perguntas:
    print(f"\n👤 Doador: {pergunta}")
    resposta = chat_assistente(pergunta)
    print(f"🤖 Assistente: {resposta}")

Chat iniciado com contexto do pão fresco (Teste 1)

👤 Doador: Posso embalar em saco plástico comum?
🤖 Assistente: Sim, pode usar saco plástico comum, desde que esteja limpo, novo e bem vedado. Isso ajuda a manter a maciez do pão integral e evita qualquer contaminação até a entrega à ONG.

👤 Doador: O receptor pode buscar em até 3 horas?
🤖 Assistente: Sim, o prazo de 3 horas é excelente e totalmente seguro para a retirada. Mantenha o pão bem fechado até a chegada do receptor para garantir a qualidade da doação.

👤 Doador: Preciso colocar alguma etiqueta de identificação?
🤖 Assistente: Sim, é recomendado colocar uma etiqueta simples com o **nome do alimento** e a **data de hoje**. Isso ajuda a ONG a organizar a distribuição com total segurança!


In [31]:
# Célula 8 – Resumo dos resultados (evidência de avaliação)
print("\n" + "="*55)
print("  RESUMO DOS TESTES – SobraBoa IA")
print("="*55)

testes = [
    ("Pão francês fresco",    resultado_pao),
    ("Alimento com mofo",     resultado_mofado),
    ("Queijo / laticínio",    resultado_queijo),
]

for nome, resultado in testes:
    status = "✅ Aprovado" if resultado["aprovado"] else "❌ Recusado"
    refrig = "⚠️ Sim" if resultado["alerta_cadeia_fria"] else "Não"
    validade = resultado.get("validade_estimada") or resultado.get("validade_estimada_dias", "N/A")
    print(f"\n{nome}")
    print(f"  Status:    {status}")
    print(f"  Categoria: {resultado['categoria']}")
    print(f"  Refrig.:   {refrig}")
    print(f"  Validade:  {validade}")

print("\n" + "="*55)
print("  Técnicas aplicadas:")
print("   ✓ Visão Computacional (análise de imagem)")
print("   ✓ IA Generativa (chat contextual)")
print("   ✓ RAG básico (system prompt com regras do domínio)")
print(f"  Modelo: gemini-3.5-flash-lite")
print("="*55)


  RESUMO DOS TESTES – SobraBoa IA

Pão francês fresco
  Status:    ✅ Aprovado
  Categoria: Pães e Confeitaria
  Refrig.:   Não
  Validade:  2 a 3 dias

Alimento com mofo
  Status:    ❌ Recusado
  Categoria: Hortifrúti (frutas, legumes, verduras)
  Refrig.:   Não
  Validade:  0

Queijo / laticínio
  Status:    ✅ Aprovado
  Categoria: Laticínios
  Refrig.:   ⚠️ Sim
  Validade:  2 dias

  Técnicas aplicadas:
   ✓ Visão Computacional (análise de imagem)
   ✓ IA Generativa (chat contextual)
   ✓ RAG básico (system prompt com regras do domínio)
  Modelo: gemini-3.5-flash-lite
